# Mammography — post hoc TTA and ensembling

Exploratory test-time augmentation, seed ensembling, and post-processing using the previously trained mammography models.


In [ ]:

from pathlib import Path
import numpy as np

# Leave the dataset input as None for automatic discovery under /kaggle/input.
# Accepted: extracted ROI_Crops_256_v1 folder OR ROI_Crops_256_v1_Kaggle.zip.
SOURCE_PATH = None

# Leave the checkpoint root as None for automatic discovery under /kaggle/input and /kaggle/working.
CKPT_DIR = None

# Keep the previous training-results folder read-only.
PREVIOUS_TRAINING_OUT_DIR = Path("/kaggle/working/ROI256_TRAINING_RESULTS")

OUT_DIR = Path("/kaggle/working/ROI256_WAVE1_RESULTS")
CLEAN_WAVE1_OUT = True

MODELS_TO_RUN = ["unet", "attention_unet", "swin_tiny_unet"]
TRAINING_SEEDS = [42, 123, 2025]
SPLITS_TO_EVALUATE = ["validation", "test", "external_inbreast"]

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
AMP_INFERENCE = True
DEVICE = "cuda"  # Set the device to "cuda" or "cpu".

# Validation-only threshold sweep.
THRESHOLDS = [round(float(x), 3) for x in np.arange(0.05, 0.96, 0.05)]

USE_TTA = True

# Enable probability-map export only when sufficient disk space is available.
# False by default; enable only for re-analysis without re-inference.
SAVE_PROBABILITY_CACHE = False
PROBABILITY_DTYPE = "float32"  # Keep float32 for strict numeric reproducibility.

BASELINE_REFERENCE = {
    "test": {"dice": 0.888, "iou": 0.809, "hd95": 13.984, "asd": 5.017},
    "external_inbreast": {"dice": 0.849, "iou": 0.755, "hd95": 17.931, "asd": 6.476},
}

print("Wave 1 configuration loaded.")
print("OUT_DIR:", OUT_DIR)


In [ ]:

import os, sys, json, math, time, random, shutil, zipfile, hashlib, warnings, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    MATPLOTLIB_AVAILABLE = False
    print("matplotlib unavailable:", repr(e))

try:
    from scipy.ndimage import (
        binary_erosion,
        distance_transform_edt,
        binary_fill_holes,
        binary_closing,
        label as scipy_label,
    )
    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    print("scipy unavailable: HD95/ASD/post-processing will be limited", repr(e))

try:
    import torchvision
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_AVAILABLE = False
    print("torchvision unavailable: SwinTinyUNet cannot be built", repr(e))

if OUT_DIR.resolve() == PREVIOUS_TRAINING_OUT_DIR.resolve():
    raise RuntimeError("OUT_DIR must be different from PREVIOUS_TRAINING_OUT_DIR to preserve training outputs.")

if CLEAN_WAVE1_OUT and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
for d in ["logs", "metrics", "probabilities", "figures", "zips", "_extracted"]:
    (OUT_DIR / d).mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torchvision:", getattr(torchvision, "__version__", "NA") if TORCHVISION_AVAILABLE else "NA")
print("SciPy:", SCIPY_AVAILABLE)
print("Previous training output exists:", PREVIOUS_TRAINING_OUT_DIR.exists(), PREVIOUS_TRAINING_OUT_DIR)
print("Wave 1 output folder:", OUT_DIR)


In [ ]:

def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

seed_everything(42)


In [ ]:

def find_manifest_in_folder(folder: Path):
    hits = list(Path(folder).rglob("roi_crop_manifest.csv"))
    if not hits:
        return None
    return sorted(hits, key=lambda p: len(p.parts))[0]


def zip_contains_manifest(zip_path: Path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            return any(Path(n).name == "roi_crop_manifest.csv" for n in z.namelist())
    except Exception:
        return False


def print_input_tree(max_items=180):
    root = Path("/kaggle/input")
    print("Preview of /kaggle/input")
    if not root.exists():
        print("  /kaggle/input does not exist")
        return
    items = list(root.rglob("*"))
    for p in items[:max_items]:
        print(" ", p)
    if len(items) > max_items:
        print(" ...", len(items) - max_items, "additional items")


def resolve_source_path(source_path=None):
    if source_path is not None:
        p = Path(source_path)
        if not p.exists():
            raise FileNotFoundError(f"SOURCE_PATH not found: {p}")
        return p

    input_root = Path("/kaggle/input")
    manifests = list(input_root.rglob("roi_crop_manifest.csv"))
    if manifests:
        return sorted(manifests, key=lambda p: len(p.parts))[0].parent

    zips = sorted(input_root.rglob("*.zip"))
    candidates = [z for z in zips if zip_contains_manifest(z)]
    if candidates:
        return candidates[0]

    print_input_tree()
    raise FileNotFoundError(
        "No ROI_Crops_256 ZIP or folder containing roi_crop_manifest.csv was found. "
        "Add the ROI_Crops_256_v1 Kaggle dataset or set SOURCE_PATH manually."
    )


def prepare_dataset_root(source_path):
    source_path = Path(source_path)
    if source_path.is_dir():
        manifest_path = find_manifest_in_folder(source_path)
        if manifest_path is None:
            raise FileNotFoundError(f"roi_crop_manifest.csv not found in {source_path}")
        return manifest_path.parent

    if source_path.suffix.lower() == ".zip":
        extract_dir = OUT_DIR / "_extracted" / "ROI_Crops_256_v1"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        print("Extracting dataset ZIP:", source_path)
        with zipfile.ZipFile(source_path, "r") as z:
            bad = z.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupted ZIP member: {bad}")
            z.extractall(extract_dir)
        manifest_path = find_manifest_in_folder(extract_dir)
        if manifest_path is None:
            raise FileNotFoundError("roi_crop_manifest.csv not found after extraction")
        return manifest_path.parent

    raise ValueError(f"Unsupported source: {source_path}")

SOURCE = resolve_source_path(SOURCE_PATH)
DATA_ROOT = prepare_dataset_root(SOURCE)
manifest = pd.read_csv(DATA_ROOT / "roi_crop_manifest.csv")

print("SOURCE:", SOURCE)
print("DATA_ROOT:", DATA_ROOT)
print("Manifest shape:", manifest.shape)
display(manifest.head())

source_audit = {"source": str(SOURCE), "data_root": str(DATA_ROOT)}
if SOURCE.is_file():
    source_audit["source_sha256"] = sha256_file(SOURCE)
pd.DataFrame([source_audit]).to_csv(OUT_DIR / "logs" / "roi256_source_audit.csv", index=False)


In [ ]:

required_cols = [
    "sample_id", "dataset", "source", "split", "patient_id", "case_id", "laterality",
    "oracle_crop_flag", "lesion_ratio_original", "lesion_ratio_crop",
    "bbox_w", "bbox_h", "crop_size_native", "crop_touches_border", "npz_path"
]
missing = [c for c in required_cols if c not in manifest.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

display(manifest.groupby(["dataset", "split"]).size().reset_index(name="n"))

cbis = manifest[manifest["dataset"].astype(str).str.upper().str.contains("CBIS")].copy()
inb = manifest[manifest["dataset"].astype(str).str.upper().str.contains("INBREAST")].copy()

if cbis.empty:
    raise RuntimeError("No CBIS-DDSM sample found in the manifest")
if not {"train", "validation", "test"}.issubset(set(cbis["split"].astype(str))):
    raise RuntimeError("CBIS-DDSM must contain train/validation/test splits")

if not inb.empty and not set(inb["split"].astype(str)).issubset({"external_inbreast"}):
    raise RuntimeError("INbreast must be assigned only to external_inbreast")

split_patients = {
    s: set(cbis.loc[cbis["split"].astype(str).eq(s), "patient_id"].astype(str))
    for s in ["train", "validation", "test"]
}
for a in split_patients:
    for b in split_patients:
        if a < b:
            inter = split_patients[a] & split_patients[b]
            if inter:
                raise RuntimeError(f"CBIS patient leakage between {a} and {b}: {len(inter)} patients")
print("CBIS patient-level leakage check: PASSED")

missing_npz = []
for rel in manifest["npz_path"].astype(str):
    if not (DATA_ROOT / rel).exists():
        missing_npz.append(rel)
if missing_npz:
    raise FileNotFoundError(f"Missing NPZ files: {len(missing_npz)} examples={missing_npz[:5]}")
print("Manifest NPZ files: PASSED")


In [ ]:

class ROI256Dataset(Dataset):
    def __init__(self, df, root, image_size=256):
        self.df = df.reset_index(drop=True).copy()
        self.root = Path(root)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with np.load(self.root / row["npz_path"], allow_pickle=False) as z:
            image = z["image"].astype(np.float32)
            mask = z["mask"].astype(np.uint8)

        image = np.squeeze(image)
        mask = np.squeeze(mask)
        image = np.clip(image, 0, 1).astype(np.float32)
        mask = (mask > 0).astype(np.float32)

        image_t = torch.from_numpy(image).unsqueeze(0).float()
        mask_t = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_t,
            "mask": mask_t,
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "dataset": str(row["dataset"]),
            "split": str(row["split"]),
        }


def get_split_df(split_name):
    if split_name in ["validation", "test"]:
        return manifest[
            manifest["dataset"].astype(str).str.upper().str.contains("CBIS")
            & manifest["split"].astype(str).eq(split_name)
        ].copy()
    if split_name == "external_inbreast":
        return manifest[
            manifest["dataset"].astype(str).str.upper().str.contains("INBREAST")
            & manifest["split"].astype(str).eq("external_inbreast")
        ].copy()
    raise ValueError(split_name)


def make_eval_loaders(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    loaders = {}
    dfs = {}
    for split in SPLITS_TO_EVALUATE:
        df = get_split_df(split)
        dfs[split] = df
        loaders[split] = DataLoader(
            ROI256Dataset(df, DATA_ROOT, image_size=IMAGE_SIZE),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            worker_init_fn=worker_init_fn,
            generator=g,
        )
    return loaders, dfs

loaders, split_dfs = make_eval_loaders(42)
for split, loader in loaders.items():
    print(split, len(loader.dataset), "samples")
b = next(iter(loaders["validation"]))
print("Sanity batch:", b["image"].shape, b["mask"].shape, float(b["image"].min()), float(b["image"].max()))


In [ ]:

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        groups = min(groups, out_ch)
        while out_ch % groups != 0 and groups > 1:
            groups -= 1
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.e4 = ConvBlock(base*4, base*8)
        self.b = ConvBlock(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.g = nn.Conv2d(g_ch, inter_ch, 1)
        self.x = nn.Conv2d(x_ch, inter_ch, 1)
        self.psi = nn.Sequential(nn.SiLU(inplace=True), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())
    def forward(self, g, x):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return x * self.psi(self.g(g) + self.x(x))

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.e4 = ConvBlock(base*4, base*8)
        self.b = ConvBlock(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.a4 = AttentionGate(base*8, base*8, base*4)
        self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.a3 = AttentionGate(base*4, base*4, base*2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.a2 = AttentionGate(base*2, base*2, base)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.a1 = AttentionGate(base, base, max(base//2, 1))
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        u4 = self.u4(b); d4 = self.d4(torch.cat([u4, self.a4(u4, e4)], dim=1))
        u3 = self.u3(d4); d3 = self.d3(torch.cat([u3, self.a3(u3, e3)], dim=1))
        u2 = self.u2(d3); d2 = self.d2(torch.cat([u2, self.a2(u2, e2)], dim=1))
        u1 = self.u1(d2); d1 = self.d1(torch.cat([u1, self.a1(u1, e1)], dim=1))
        return self.out(d1)

class UpConv(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_AVAILABLE:
            raise RuntimeError("torchvision is required for SwinTinyUNet")
        from torchvision.models import swin_t
        self.in_adapter = nn.Conv2d(1, 3, 1, bias=False)
        with torch.no_grad():
            self.in_adapter.weight.fill_(1.0)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.dec3 = UpConv(512, 384, 256)
        self.dec2 = UpConv(256, 192, 128)
        self.dec1 = UpConv(128, 96, 64)
        self.up0a = nn.ConvTranspose2d(64, 32, 2, 2)
        self.c0a = ConvBlock(32, 32)
        self.up0b = nn.ConvTranspose2d(32, 16, 2, 2)
        self.c0b = ConvBlock(16, 16)
        self.out = nn.Conv2d(16, out_ch, 1)
    def _nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96, 192, 384, 768]:
            return x.permute(0, 3, 1, 2).contiguous()
        return x
    def forward(self, x):
        y = self.in_adapter(x)
        y = (y - self.mean) / self.std
        feats = []
        for i, layer in enumerate(self.features):
            y = layer(y)
            if i in [1, 3, 5, 7]:
                feats.append(self._nchw(y))
        if len(feats) != 4:
            raise RuntimeError(f"Unexpected Swin feature count: {len(feats)}")
        s1, s2, s3, s4 = feats
        x = self.center(s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.c0a(self.up0a(x))
        x = self.c0b(self.up0b(x))
        if x.shape[-2:] != (IMAGE_SIZE, IMAGE_SIZE):
            x = F.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        return self.out(x)


def build_model(name):
    name = name.lower()
    if name == "unet":
        return UNet()
    if name == "attention_unet":
        return AttentionUNet()
    if name == "swin_tiny_unet":
        return SwinTinyUNet()
    raise ValueError(name)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

device = torch.device(DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu")
print("Using device:", device)
for name in MODELS_TO_RUN:
    m = build_model(name).to(device)
    with torch.no_grad():
        y = m(torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device))
    print(name, "params", count_params(m), "out", tuple(y.shape))
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:

def binary_metrics(pred, target, eps=1e-7):
    pred = pred.astype(bool)
    target = target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    p = pred.sum()
    t = target.sum()
    dice = (2*tp + eps) / (p + t + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    iou = (tp + eps) / (tp + fp + fn + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps) if t > 0 else np.nan
    tn = np.logical_and(~pred, ~target).sum()
    return {
        "dice": float(dice), "iou": float(iou), "precision": float(precision), "recall": float(recall),
        "empty_pred": int(p == 0), "empty_target": int(t == 0),
        "pred_area": int(p), "target_area": int(t),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def hd95_asd(pred, target):
    if not SCIPY_AVAILABLE:
        return {"hd95": np.nan, "asd": np.nan}
    pred = pred.astype(bool)
    target = target.astype(bool)
    if pred.sum() == 0 or target.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    pb = pred ^ binary_erosion(pred)
    tb = target ^ binary_erosion(target)
    if pb.sum() == 0 or tb.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    dt_t = distance_transform_edt(~tb)
    dt_p = distance_transform_edt(~pb)
    d = np.concatenate([dt_t[pb], dt_p[tb]]).astype(np.float32)
    return {"hd95": float(np.percentile(d, 95)), "asd": float(d.mean())}


def largest_connected_component(mask):
    if not SCIPY_AVAILABLE:
        return mask.astype(np.uint8)
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return mask.astype(np.uint8)
    lab, n = scipy_label(mask)
    if n <= 1:
        return mask.astype(np.uint8)
    counts = np.bincount(lab.ravel())
    counts[0] = 0
    keep = counts.argmax()
    return (lab == keep).astype(np.uint8)


def postprocess_mask(pred):
    pred = pred.astype(bool)
    if pred.sum() == 0 or not SCIPY_AVAILABLE:
        return pred.astype(np.uint8)
    pred = binary_fill_holes(pred)
    pred = binary_closing(pred, structure=np.ones((3, 3), dtype=bool), iterations=1)
    pred = largest_connected_component(pred)
    return pred.astype(np.uint8)


def aggregate_from_records(records):
    df = pd.DataFrame(records)
    result = {"n": int(len(df))}
    for c in ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred"]:
        result[c] = float(np.nanmean(df[c])) if c in df.columns and len(df) else np.nan
    return result


def evaluate_prob_maps(rows, probs, masks, threshold, config_name, split_name, postprocess=False, full_surface=True):
    records = []
    for row, prob, mask in zip(rows, probs, masks):
        pred = (prob >= threshold).astype(np.uint8)
        if postprocess:
            pred = postprocess_mask(pred)
        met = binary_metrics(pred, mask)
        surf = hd95_asd(pred, mask) if full_surface else {"hd95": np.nan, "asd": np.nan}
        records.append({
            **row,
            "config": config_name,
            "split_eval": split_name,
            "threshold": float(threshold),
            "postprocess": bool(postprocess),
            **met,
            **surf,
        })
    return pd.DataFrame(records)


def validation_score_for_threshold(probs, masks, threshold, postprocess=False):
    vals = []
    ious = []
    for prob, mask in zip(probs, masks):
        pred = (prob >= threshold).astype(np.uint8)
        if postprocess:
            pred = postprocess_mask(pred)
        met = binary_metrics(pred, mask)
        vals.append(met["dice"])
        ious.append(met["iou"])
    return float(np.nanmean(vals)), float(np.nanmean(ious))


In [ ]:

def zip_contains_checkpoints(zip_path: Path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            return any(n.endswith("_best.pt") or n.endswith(".pth") or n.endswith(".ckpt") for n in z.namelist())
    except Exception:
        return False


_EXTRACTED_CKPT_ROOTS = None

def extract_checkpoint_zips_if_needed():
    global _EXTRACTED_CKPT_ROOTS
    if _EXTRACTED_CKPT_ROOTS is not None:
        return _EXTRACTED_CKPT_ROOTS
    extracted_roots = []
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        _EXTRACTED_CKPT_ROOTS = extracted_roots
        return extracted_roots
    for zpath in sorted(input_root.rglob("*.zip")):
        if not zip_contains_checkpoints(zpath):
            continue
        dest = OUT_DIR / "_extracted" / f"checkpoints_from_{zpath.stem}"
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting checkpoint-like files from:", zpath)
        with zipfile.ZipFile(zpath, "r") as z:
            for name in z.namelist():
                if name.endswith((".pt", ".pth", ".ckpt")):
                    z.extract(name, dest)
        extracted_roots.append(dest)
    _EXTRACTED_CKPT_ROOTS = extracted_roots
    return extracted_roots


def checkpoint_search_roots():
    roots = []
    if CKPT_DIR is not None:
        roots.append(Path(CKPT_DIR))
    roots += [
        PREVIOUS_TRAINING_OUT_DIR / "checkpoints",
        PREVIOUS_TRAINING_OUT_DIR,
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]
    roots += extract_checkpoint_zips_if_needed()
    out = []
    seen = set()
    for r in roots:
        if r.exists():
            key = str(r.resolve())
            if key not in seen:
                out.append(r)
                seen.add(key)
    return out


def find_checkpoint(model_name, seed):
    patterns = [
        f"{model_name}_seed{seed}_best.pt",
        f"{model_name}_seed{seed}_best.pth",
        f"{model_name}_seed{seed}_best.ckpt",
        f"*{model_name}*seed{seed}*.pt",
        f"*{model_name}*seed{seed}*.pth",
        f"*{model_name}*seed{seed}*.ckpt",
    ]
    candidates = []
    for root in checkpoint_search_roots():
        for pat in patterns:
            candidates.extend(root.rglob(pat))
    candidates = [p for p in candidates if p.is_file()]
    exact_name = f"{model_name}_seed{seed}_best.pt"
    candidates = sorted(candidates, key=lambda p: (p.name != exact_name, len(p.parts), str(p)))
    if not candidates:
        raise FileNotFoundError(
            f"Checkpoint not found for {model_name} seed={seed}. "
            "Attach the previous training notebook output or set CKPT_DIR manually."
        )
    return candidates[0]


def load_model_strict(model_name, seed, device):
    ckpt_path = find_checkpoint(model_name, seed)
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        state_dict = ckpt
    else:
        raise RuntimeError(f"Unsupported checkpoint format: {ckpt_path}")

    model = build_model(model_name).to(device)
    # Keep strict=True; do not switch to strict=False.
    incompatible = model.load_state_dict(state_dict, strict=True)
    missing = getattr(incompatible, "missing_keys", [])
    unexpected = getattr(incompatible, "unexpected_keys", [])
    if missing or unexpected:
        raise RuntimeError(
            f"Strict state_dict mismatch for {model_name} seed={seed}: "
            f"missing={missing}, unexpected={unexpected}"
        )
    model.eval()
    print(f"Loaded strictly: {model_name} seed={seed} <- {ckpt_path}")
    return model, ckpt_path

ckpt_rows = []
for model_name in MODELS_TO_RUN:
    for seed in TRAINING_SEEDS:
        p = find_checkpoint(model_name, seed)
        ckpt_rows.append({"model": model_name, "seed": seed, "checkpoint_path": str(p), "sha256": sha256_file(p)})
ckpt_audit = pd.DataFrame(ckpt_rows)
ckpt_audit.to_csv(OUT_DIR / "logs" / "checkpoint_audit.csv", index=False)
display(ckpt_audit)


In [ ]:

def tta_transforms():
    return [
        ("identity", lambda x: x, lambda y: y),
        ("hflip", lambda x: torch.flip(x, dims=[-1]), lambda y: torch.flip(y, dims=[-1])),
        ("vflip", lambda x: torch.flip(x, dims=[-2]), lambda y: torch.flip(y, dims=[-2])),
        ("rot90", lambda x: torch.rot90(x, k=1, dims=(-2, -1)), lambda y: torch.rot90(y, k=-1, dims=(-2, -1))),
        ("rot180", lambda x: torch.rot90(x, k=2, dims=(-2, -1)), lambda y: torch.rot90(y, k=-2, dims=(-2, -1))),
        ("rot270", lambda x: torch.rot90(x, k=3, dims=(-2, -1)), lambda y: torch.rot90(y, k=-3, dims=(-2, -1))),
    ]


@torch.no_grad()
def collect_probs_tta(model, loader, device, use_tta=True):
    model.eval()
    rows, probs, masks = [], [], []
    transforms = tta_transforms() if use_tta else [("identity", lambda x: x, lambda y: y)]

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        acc = None
        for name, fwd, inv in transforms:
            xt = fwd(x)
            with autocast(enabled=AMP_INFERENCE and device.type == "cuda"):
                logits = model(xt)
                pr = torch.sigmoid(logits)
            pr = inv(pr)
            acc = pr if acc is None else acc + pr
        acc = acc / float(len(transforms))
        pr_np = acc.detach().cpu().numpy()
        ma_np = batch["mask"].cpu().numpy()
        for i in range(x.shape[0]):
            rows.append({
                "sample_id": batch["sample_id"][i],
                "patient_id": batch["patient_id"][i],
                "dataset": batch["dataset"][i],
                "split": batch["split"][i],
            })
            probs.append(pr_np[i, 0].astype(np.float32))
            masks.append(ma_np[i, 0].astype(np.uint8))
    return rows, np.stack(probs).astype(np.float32), np.stack(masks).astype(np.uint8)


def save_probability_array(split, key, arr):
    if not SAVE_PROBABILITY_CACHE:
        return None
    out = OUT_DIR / "probabilities" / split
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{key}.npy"
    np.save(path, arr.astype(PROBABILITY_DTYPE, copy=False))
    return path


In [ ]:

all_probs = {split: {} for split in SPLITS_TO_EVALUATE}
split_rows = {}
split_masks = {}

for model_name in MODELS_TO_RUN:
    for seed in TRAINING_SEEDS:
        key = f"{model_name}_seed{seed}_tta" if USE_TTA else f"{model_name}_seed{seed}"
        print(f"\n===== Inference {key} | {now()} =====")
        seed_everything(seed)
        model, ckpt_path = load_model_strict(model_name, seed, device)

        for split in SPLITS_TO_EVALUATE:
            rows, probs, masks = collect_probs_tta(model, loaders[split], device, use_tta=USE_TTA)
            sample_ids = [r["sample_id"] for r in rows]
            if split not in split_rows:
                split_rows[split] = rows
                split_masks[split] = masks
            else:
                ref_ids = [r["sample_id"] for r in split_rows[split]]
                if sample_ids != ref_ids:
                    raise RuntimeError(f"Sample order mismatch for split={split}, key={key}")
                if not np.array_equal(masks, split_masks[split]):
                    raise RuntimeError(f"Mask mismatch for split={split}, key={key}")
            all_probs[split][key] = probs
            pth = save_probability_array(split, key, probs)
            print(split, key, probs.shape, "saved=" + str(pth) if pth else "")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Inference complete.")
for split in SPLITS_TO_EVALUATE:
    print(split, "configs:", list(all_probs[split].keys()))


In [ ]:

def mean_probs(split, keys):
    arrs = [all_probs[split][k] for k in keys]
    return np.mean(np.stack(arrs, axis=0), axis=0).astype(np.float32)

ensemble_defs = {}
for model_name in MODELS_TO_RUN:
    keys = [f"{model_name}_seed{seed}_tta" if USE_TTA else f"{model_name}_seed{seed}" for seed in TRAINING_SEEDS]
    ensemble_defs[f"{model_name}_ens3_tta"] = keys

all_single_keys = []
for model_name in MODELS_TO_RUN:
    for seed in TRAINING_SEEDS:
        all_single_keys.append(f"{model_name}_seed{seed}_tta" if USE_TTA else f"{model_name}_seed{seed}")
ensemble_defs["ensemble_all9_tta"] = all_single_keys

for split in SPLITS_TO_EVALUATE:
    for ens_name, keys in ensemble_defs.items():
        all_probs[split][ens_name] = mean_probs(split, keys)
        pth = save_probability_array(split, ens_name, all_probs[split][ens_name])
        print(split, ens_name, all_probs[split][ens_name].shape, "saved=" + str(pth) if pth else "")

config_names = list(all_probs["validation"].keys())
print("Total configurations evaluated:", len(config_names))
print(config_names)


In [ ]:

def select_threshold_for_config(config_name, postprocess=False):
    probs = all_probs["validation"][config_name]
    masks = split_masks["validation"]
    rows = []
    for th in THRESHOLDS:
        dice, iou = validation_score_for_threshold(probs, masks, th, postprocess=postprocess)
        rows.append({
            "config": config_name,
            "postprocess": bool(postprocess),
            "threshold": float(th),
            "val_dice": dice,
            "val_iou": iou,
            "dist_to_0_5": abs(float(th) - 0.5),
        })
    sweep = pd.DataFrame(rows)
    ranked = sweep.sort_values(
        ["val_dice", "val_iou", "dist_to_0_5", "threshold"],
        ascending=[False, False, True, True],
    )
    best_th = float(ranked.iloc[0]["threshold"])
    return best_th, sweep.sort_values("threshold")

sweep_parts = []
selected_thresholds = []
for config_name in config_names:
    for post in [False, True]:
        th, sweep = select_threshold_for_config(config_name, postprocess=post)
        sweep_parts.append(sweep)
        best_row = sweep.loc[sweep["threshold"].eq(th)].iloc[0].to_dict()
        selected_thresholds.append({
            "config": config_name,
            "postprocess": bool(post),
            "selected_threshold": th,
            "selected_on": "CBIS-DDSM validation only",
            "selected_val_dice": best_row["val_dice"],
            "selected_val_iou": best_row["val_iou"],
        })
        print(f"{config_name:35s} post={post:<5} threshold={th:.2f} val_dice={best_row['val_dice']:.4f}")

threshold_sweeps = pd.concat(sweep_parts, ignore_index=True)
threshold_sweeps.to_csv(OUT_DIR / "metrics" / "wave1_threshold_sweeps_validation.csv", index=False)
selected_thresholds_df = pd.DataFrame(selected_thresholds)
selected_thresholds_df.to_csv(OUT_DIR / "metrics" / "wave1_selected_thresholds.csv", index=False)
display(selected_thresholds_df.sort_values(["postprocess", "selected_val_dice"], ascending=[True, False]).head(20))


In [ ]:

detailed_parts = []
summary_rows = []

for _, sel in selected_thresholds_df.iterrows():
    config_name = sel["config"]
    post = bool(sel["postprocess"])
    th = float(sel["selected_threshold"])
    for split in SPLITS_TO_EVALUATE:
        df_det = evaluate_prob_maps(
            split_rows[split],
            all_probs[split][config_name],
            split_masks[split],
            threshold=th,
            config_name=config_name,
            split_name=split,
            postprocess=post,
            full_surface=True,
        )
        detailed_parts.append(df_det)
        agg = aggregate_from_records(df_det.to_dict("records"))
        row = {
            "config": config_name,
            "postprocess": post,
            "split": split,
            "selected_threshold": th,
            "threshold_selected_on": "CBIS-DDSM validation only",
            **agg,
        }
        if split in BASELINE_REFERENCE:
            row["baseline_dice_ref"] = BASELINE_REFERENCE[split]["dice"]
            row["delta_dice_vs_baseline"] = row["dice"] - BASELINE_REFERENCE[split]["dice"]
            row["baseline_hd95_ref"] = BASELINE_REFERENCE[split]["hd95"]
            row["delta_hd95_vs_baseline"] = row["hd95"] - BASELINE_REFERENCE[split]["hd95"]
        summary_rows.append(row)
        print(config_name, "post=", post, "split=", split, "dice=", round(row["dice"], 4), "hd95=", round(row["hd95"], 3) if not np.isnan(row["hd95"]) else np.nan)

wave1_detailed = pd.concat(detailed_parts, ignore_index=True)
wave1_summary = pd.DataFrame(summary_rows)

wave1_detailed.to_csv(OUT_DIR / "metrics" / "wave1_detailed_metrics.csv", index=False)
wave1_summary.to_csv(OUT_DIR / "metrics" / "wave1_results.csv", index=False)

print("Saved:", OUT_DIR / "metrics" / "wave1_results.csv")
display(wave1_summary.head())


In [ ]:

metric_cols = ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred", "selected_threshold"]
wide_parts = []
base_cols = ["config", "postprocess"]
for split in SPLITS_TO_EVALUATE:
    tmp = wave1_summary[wave1_summary["split"].eq(split)][base_cols + metric_cols].copy()
    tmp = tmp.rename(columns={c: f"{split}_{c}" for c in metric_cols})
    wide_parts.append(tmp)
from functools import reduce
wave1_wide = reduce(lambda left, right: pd.merge(left, right, on=base_cols, how="outer"), wide_parts)

wave1_wide["rank_by_validation_dice"] = wave1_wide["validation_dice"].rank(method="min", ascending=False).astype(int)
wave1_wide = wave1_wide.sort_values(["rank_by_validation_dice", "config", "postprocess"]).reset_index(drop=True)

if "test_dice" in wave1_wide.columns:
    wave1_wide["test_delta_dice_vs_swin_baseline"] = wave1_wide["test_dice"] - BASELINE_REFERENCE["test"]["dice"]
if "external_inbreast_dice" in wave1_wide.columns:
    wave1_wide["external_inbreast_delta_dice_vs_swin_baseline"] = wave1_wide["external_inbreast_dice"] - BASELINE_REFERENCE["external_inbreast"]["dice"]
if "external_inbreast_hd95" in wave1_wide.columns:
    wave1_wide["external_inbreast_delta_hd95_vs_swin_baseline"] = wave1_wide["external_inbreast_hd95"] - BASELINE_REFERENCE["external_inbreast"]["hd95"]

wave1_wide.to_csv(OUT_DIR / "metrics" / "wave1_results_wide.csv", index=False)
display(wave1_wide.head(20))

primary = wave1_wide.iloc[0].to_dict()
print("Primary configuration selected by validation Dice only:")
print(json.dumps({k: v for k, v in primary.items() if k in ["config", "postprocess", "validation_dice", "test_dice", "external_inbreast_dice", "rank_by_validation_dice"]}, indent=2, default=str))

json_payload = {
    "created_at": now(),
    "protocol_guardrail": "Thresholds and primary ranking selected on CBIS-DDSM validation only; CBIS-DDSM test and INbreast are sealed evaluations.",
    "baseline_reference": BASELINE_REFERENCE,
    "primary_by_validation": primary,
    "selected_thresholds": selected_thresholds_df.to_dict("records"),
    "summary_long": wave1_summary.to_dict("records"),
    "summary_wide": wave1_wide.to_dict("records"),
}
with open(OUT_DIR / "metrics" / "wave1_results.json", "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)

md_path = OUT_DIR / "metrics" / "wave1_results.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Wave 1 Results — ROI256 Mammography\n\n")
    f.write("## Protocol guardrail\n\n")
    f.write("Thresholds are selected only on CBIS-DDSM validation. CBIS-DDSM test and INbreast external validation are evaluated with frozen thresholds.\n\n")
    f.write("## Primary configuration selected by validation Dice\n\n")
    primary_cols = [c for c in [
        "config", "postprocess", "validation_selected_threshold", "validation_dice", "test_dice", "external_inbreast_dice",
        "test_delta_dice_vs_swin_baseline", "external_inbreast_delta_dice_vs_swin_baseline", "external_inbreast_hd95",
        "external_inbreast_delta_hd95_vs_swin_baseline"
    ] if c in wave1_wide.columns]
    f.write(wave1_wide.loc[[0], primary_cols].to_markdown(index=False))
    f.write("\n\n## All configurations, ranked by validation Dice\n\n")
    show_cols = [c for c in [
        "rank_by_validation_dice", "config", "postprocess", "validation_selected_threshold", "validation_dice",
        "test_dice", "external_inbreast_dice", "external_inbreast_hd95",
        "test_delta_dice_vs_swin_baseline", "external_inbreast_delta_dice_vs_swin_baseline",
        "external_inbreast_delta_hd95_vs_swin_baseline"
    ] if c in wave1_wide.columns]
    f.write(wave1_wide[show_cols].to_markdown(index=False))
    f.write("\n\n## Files\n\n")
    for p in ["wave1_results.csv", "wave1_results_wide.csv", "wave1_results.json", "wave1_threshold_sweeps_validation.csv", "wave1_detailed_metrics.csv"]:
        f.write(f"- `{p}`\n")

print("Saved JSON:", OUT_DIR / "metrics" / "wave1_results.json")
print("Saved Markdown:", md_path)


In [ ]:

if MATPLOTLIB_AVAILABLE:
    fig_df = wave1_wide.copy()
    # External Dice may be displayed but must not be used for primary model selection.
    fig_df = fig_df.sort_values("rank_by_validation_dice")
    labels = [f"{r.config}\npost={r.postprocess}" for _, r in fig_df.iterrows()]

    plt.figure(figsize=(max(10, 0.55 * len(labels)), 5))
    plt.bar(range(len(labels)), fig_df["external_inbreast_dice"].values)
    plt.axhline(BASELINE_REFERENCE["external_inbreast"]["dice"], linestyle="--", linewidth=1)
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel("INbreast Dice")
    plt.title("Wave 1 configurations — external Dice shown for reporting only")
    plt.tight_layout()
    fig_path = OUT_DIR / "figures" / "wave1_external_dice_ranked_by_validation.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("Saved figure:", fig_path)
else:
    print("Matplotlib unavailable; skipping figures.")


In [ ]:

zip_base = OUT_DIR
zip_path = OUT_DIR.parent / "ROI256_WAVE1_RESULTS.zip"
if zip_path.exists():
    zip_path.unlink()

# Exclude large extracted data files from the final archive.
exclude_dirs = {str((OUT_DIR / "_extracted").resolve())}
if not SAVE_PROBABILITY_CACHE:
    exclude_dirs.add(str((OUT_DIR / "probabilities").resolve()))

tmp_pack = OUT_DIR.parent / "ROI256_WAVE1_RESULTS_PACK"
if tmp_pack.exists():
    shutil.rmtree(tmp_pack)
shutil.copytree(OUT_DIR, tmp_pack, ignore=shutil.ignore_patterns("_extracted", "probabilities" if not SAVE_PROBABILITY_CACHE else "__NO_MATCH__"))
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=tmp_pack.parent, base_dir=tmp_pack.name)
shutil.rmtree(tmp_pack)

sha = sha256_file(zip_path)
pd.DataFrame([{"zip_path": str(zip_path), "sha256": sha}]).to_csv(OUT_DIR / "logs" / "ROI256_WAVE1_RESULTS_sha256.csv", index=False)

print("Final Wave 1 archive:", zip_path)
print("SHA256:", sha)
print("Key files:")
for p in [
    OUT_DIR / "metrics" / "wave1_results.csv",
    OUT_DIR / "metrics" / "wave1_results_wide.csv",
    OUT_DIR / "metrics" / "wave1_results.json",
    OUT_DIR / "metrics" / "wave1_results.md",
    OUT_DIR / "logs" / "checkpoint_audit.csv",
]:
    print(" -", p, "exists=", p.exists())
